# Fig 5: Collocated Prefill/Decode Interference

**Output:** `model_outputs/<MODEL_SHORT>/paper/section3/fig5/interference_combined.{pdf,png}`

Paths are resolved via `MODEL_DATA_DIR` from `profiling/config.py` (set `MODELS_ROOT` in `config.env` to override).

### Call order
1. `profiling/launch_interference.sh` — collect interference data (GPUs 0–2, auto-merges shards; writes to `model_outputs/<MODEL_SHORT>/paper/section3/profiling/interference_data/` and `interference_kv_data/`). Pass `1` or `2` to run only that phase (e.g. `bash launch_interference.sh 2` for KV sweep only).
2. `scripts/plot_interference.py` — parse B-sweep (Exp 2/3) → `interference_summary.csv`
3. `scripts/plot_interference_kv.py` — parse KV sweep → `interference_kv_summary.csv`
4. `scripts/plot_interference_combined.py` — reads both CSVs → combined figure

Step 1 only needs to run once. Steps 2–3 must run before step 4 (they produce the CSVs it reads).

In [1]:
import subprocess
from pathlib import Path

REPO_ROOT = next(
    p for p in [Path.cwd()] + list(Path.cwd().parents)
    if (p / ".conserve_root").exists()
)


def run(cmd):
    buf = []
    with subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    ) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
            buf.append(line)
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed (exit {proc.returncode}): {' '.join(str(c) for c in cmd)}")
    return "".join(buf)


In [ ]:
# Step 1 (phase 2 only): run KV sweep when phase 1 data already exists.
run(["bash", str(REPO_ROOT / "profiling/launch_interference.sh"), "2"])

In [2]:
# Step 1: collect raw interference data. Skip if data already exists.
run(["bash", str(REPO_ROOT / "profiling/launch_interference.sh")])

=== 2026-06-25T18:40:10-05:00 phase start: run_interference.py ===
  shard 0 (pid 445662) FAILED — see /data/projects/eicchen/conserve_project/conserve/model_outputs/Qwen3-0.6B/paper/section3/profiling/interference_data/launcher_gpu0.log
  shard 1 (pid 445663) FAILED — see /data/projects/eicchen/conserve_project/conserve/model_outputs/Qwen3-0.6B/paper/section3/profiling/interference_data/launcher_gpu1.log
  shard 2 (pid 445664) FAILED — see /data/projects/eicchen/conserve_project/conserve/model_outputs/Qwen3-0.6B/paper/section3/profiling/interference_data/launcher_gpu2.log
=== 2026-06-25T20:21:19-05:00 ERROR: a shard of run_interference.py failed ===


RuntimeError: Command failed (exit 1): bash /data/projects/eicchen/conserve_project/conserve/profiling/launch_interference.sh

In [ ]:
# Steps 2-4: plot
%matplotlib inline
%run ../scripts/plot_interference.py
%run ../scripts/plot_interference_kv.py
%run ../scripts/plot_interference_combined.py

In [ ]:
from IPython.display import Image
import sys; sys.path.insert(0, str(REPO_ROOT / "profiling"))
from config import MODEL_SHORT

Image(str(REPO_ROOT / "paper/figures/section3/output" / MODEL_SHORT / "300W" / "interference_combined.png"))